In [1]:
import pandas as pd
import sklearn.metrics
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
results = pd.read_csv('../../result/codellama-13B_2024cve_processed.csv')

print(f"ROC-AUC: {sklearn.metrics.roc_auc_score(results.vul_label, results.new_score)}")

MFRs = []
def calculate_ranked_metrics(df):
    # MFRs = []
    global MFRs

    df_grouped = df.groupby(['file','function_name'])
    for name, group in df_grouped:
        group = group.sample(frac=1, random_state=0).reset_index(drop=True)
        if group.vul_label.sum() == 0:
            continue
        
        group = group.sort_values("new_score", ascending=False)
        group = group.reset_index(drop=True)

        for (idx, line) in group.iterrows():
            if line.vul_label == 1:
                MFRs.append(idx)
                break

    MFR = np.mean(MFRs)
    N_MFR = MFR / 76
    Top_1 = (np.array(MFRs) <= 0).sum() / len(MFRs)
    Top_3 = (np.array(MFRs) <= 2).sum() / len(MFRs)
    Top_5 = (np.array(MFRs) <= 4).sum() / len(MFRs)
    Top_10 = (np.array(MFRs) <= 9).sum() / len(MFRs)

    return MFR, N_MFR, Top_1, Top_3, Top_5, Top_10

print("Magma Metrics:", calculate_ranked_metrics(results))

ROC-AUC: 0.6857144845734757
Magma Metrics: (16.71917808219178, 0.21998918529199712, 0.1643835616438356, 0.3356164383561644, 0.4726027397260274, 0.6643835616438356)
